In [0]:
%sql
-- =========================================================================
-- 1. CAPA BRONZE DECLARATIVA: Streaming Tables desde SQL Server (erp_landing)
-- =========================================================================
CREATE OR REFRESH STREAMING TABLE logistics_prod.bronze.erp_fleet_assets_raw
COMMENT "Ingesta incremental desde SQL Server (dbo.FleetAssets) vía Unity Catalog Volume"
AS
SELECT
    *,
    current_timestamp() AS _ingested_at,
    _metadata.file_path AS _source_file
FROM STREAM read_files(
    '/Volumes/logistics_prod/bronze/erp_landing/FleetAssets/',
    format => 'csv',
    header => true
);

CREATE OR REFRESH STREAMING TABLE logistics_prod.bronze.erp_maintenance_raw
COMMENT "Ingesta incremental desde SQL Server (dbo.MaintenanceLogs)"
AS
SELECT
    *,
    current_timestamp() AS _ingested_at,
    _metadata.file_path AS _source_file
FROM STREAM read_files(
    '/Volumes/logistics_prod/bronze/erp_landing/MaintenanceLogs/',
    format => 'csv',
    header => true
);

-- =========================================================================
-- 2. CAPA SILVER DECLARATIVA: Calidad con EXPECTATIONS + Deduplicación
-- =========================================================================
CREATE OR REFRESH MATERIALIZED VIEW logistics_prod.silver.erp_fleet_assets_clean (
    CONSTRAINT valid_asset_id EXPECT (asset_id IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_temp_sla EXPECT (max_allowed_temp_c BETWEEN -40 AND 50) ON VIOLATION DROP ROW
)
COMMENT "Maestro de flota limpio y deduplicado con reglas declarativas de calidad"
AS
SELECT
    UPPER(TRIM(asset_id)) AS asset_id,
    TRIM(license_plate) AS license_plate,
    TRIM(driver_name) AS driver_name,
    TRIM(cargo_category) AS cargo_category,
    CAST(max_allowed_temp_c AS DOUBLE) AS max_allowed_temp_c,
    TRIM(assigned_hub) AS assigned_hub,
    CAST(last_updated_at AS TIMESTAMP) AS last_updated_at
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY asset_id ORDER BY last_updated_at DESC) AS rn
    FROM logistics_prod.bronze.erp_fleet_assets_raw
)
WHERE rn = 1;

-- =========================================================================
-- 3. CAPA GOLD DECLARATIVA: Cruce Multi-Fuente (Sensores IoT + SQL Server ERP)
-- =========================================================================
CREATE OR REFRESH MATERIALIZED VIEW logistics_prod.gold.contractual_sla_breaches
COMMENT "Cruce en tiempo real entre telemetría IoT y contratos SLA de SQL Server"
AS
SELECT
    t.event_timestamp,
    t.asset_id,
    f.license_plate,
    f.driver_name,
    f.assigned_hub,
    f.cargo_category,
    t.temperature_c AS actual_temp_c,
    f.max_allowed_temp_c AS contract_max_temp_c,
    ROUND(t.temperature_c - f.max_allowed_temp_c, 2) AS degrees_above_sla,
    t.shipment_status,
    t.delay_reason,
    CASE
        WHEN t.temperature_c > f.max_allowed_temp_c AND f.cargo_category = 'Pharmaceuticals'
            THEN 'CRITICAL_PHARMA_SPOILAGE_RISK'
        WHEN t.temperature_c > f.max_allowed_temp_c
            THEN 'WARNING_CONTRACT_TEMP_BREACH'
        ELSE 'COMPLIANT'
    END AS sla_compliance_status
FROM logistics_prod.silver.logistics_clean t
INNER JOIN logistics_prod.silver.erp_fleet_assets_clean f
    ON t.asset_id = f.asset_id
WHERE t.temperature_c > f.max_allowed_temp_c;
